# Quickstart: Training pinnrl on a `The Well` benchmark dataset

This notebook walks through training a neural surrogate on `active_matter` from
[Polymathic AI's `the_well`](https://github.com/PolymathicAI/the_well) — 200 epochs of an FNO
in `data_only` mode (~3 minutes on CPU). The Well datasets ship without an analytical
solver, so the network learns directly from the reference field; pinnrl treats it identically
to any other observation set.

The end of the notebook saves a side-by-side prediction-vs-reference plot to
`notebooks/images/04_well_active_matter.png`, ready for figure embedding.

Prerequisites:

```bash
pip install 'pinnrl[well]'
```

If you have already downloaded the dataset locally with `the-well-download`, point `base=` at it.
Otherwise the loader streams from `hf://datasets/polymathic-ai/`.

In [1]:
import os, sys, logging

# Silence per-epoch tqdm bars and the trainer's INFO logging so the notebook reads cleanly.
os.environ["TQDM_DISABLE"] = "1"
logging.basicConfig(level=logging.WARNING, force=True)

# Make pinnrl importable when running from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

%matplotlib inline

## 1. Inspect the registry

In [2]:
from pinnrl.datasets import WELL_REGISTRY, get_entry

for name, entry in sorted(WELL_REGISTRY.items()):
    print(f"{name:35s}  {entry.n_spatial_dims}D  mode={entry.recommended_mode:14s}  arch={entry.default_architecture}")

entry = get_entry('active_matter')
print('\nactive_matter fields:', entry.fields)
print('domain:', entry.domain, 'time:', entry.time_domain)

MHD_64                               3D  mode=data_only       arch=mlp
acoustic_scattering_maze             2D  mode=data_augmented  arch=fno
active_matter                        2D  mode=data_only       arch=fno
euler_multi_quadrants_periodicBC     2D  mode=data_only       arch=fno
gray_scott_reaction_diffusion        2D  mode=data_only       arch=fno
helmholtz_staircase                  2D  mode=data_augmented  arch=fno
planetswe                            2D  mode=data_only       arch=fno
rayleigh_benard                      2D  mode=data_only       arch=fno
rayleigh_taylor_instability          3D  mode=data_only       arch=mlp
shear_flow                           2D  mode=data_only       arch=fno
turbulent_radiative_layer_2D         2D  mode=data_only       arch=fno
viscoelastic_instability             2D  mode=data_only       arch=fno

active_matter fields: ('concentration', 'velocity_x', 'velocity_y', 'orientation_xx', 'orientation_xy', 'orientation_yx', 'orientation_yy', 'strain

## 2. Load a small slice into memory

`load_well_slice` flattens the trajectory grid into the same `(x, t, u)` triple pinnrl uses everywhere — so the rest of the framework treats it identically to synthetic observations.

In [3]:
from pinnrl.datasets import load_well_slice

slice_ = load_well_slice(
    name='active_matter',
    split='train',
    n_traj=2,
    n_points=5_000,
    seed=0,
    device='cpu',
)
for k, v in slice_.items():
    print(f'{k}: shape={tuple(v.shape)}  dtype={v.dtype}')

x: shape=(5000, 2)  dtype=torch.float32
t: shape=(5000, 1)  dtype=torch.float32
u: shape=(5000, 11)  dtype=torch.float32


## 3. Train an FNO in data_only mode

The fastest path is the CLI: `build_config_dict` overlays the registry defaults and the trainer takes it from there.

In [4]:
import yaml, torch
from pathlib import Path
from pinnrl.training.train import build_config_dict, run_training

yaml_cfg = yaml.safe_load(Path("../pinnrl/config/config.yaml").read_text())
yaml_cfg.setdefault("training", {})["num_epochs"] = 200
# `_apply_well_dataset_defaults` only fills in mode when one isn't already set;
# the yaml ships with mode=forward so we override explicitly to honour the
# registry's recommendation.
yaml_cfg["training"]["mode"] = "data_only"

config_dict = build_config_dict(
    yaml_cfg,
    pde_name="Heat Equation",  # Placeholder — overridden by dataset defaults.
    arch_type="fno",
    use_rl=False,
    epochs=200,
    dataset={
        "name": "active_matter",
        "split": "train",
        "n_traj": 2,
        "n_points": 5_000,
        "seed": 0,
        "base": None,
        "use_defaults": True,
    },
)
print("mode:", config_dict["training"]["mode"])
print("input_dim/output_dim:", config_dict["model"]["input_dim"], "/", config_dict["model"]["output_dim"])

device = torch.device("cpu")
config_dict["device"] = str(device)
run_training(config_dict, device)

mode: data_only
input_dim/output_dim: 3 / 11
Experiment: 20260503_233514_active_matter_fno_no_rl
Directory: experiments/20260503_233514_active_matter_fno_no_rl


Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



ERROR:root:Error in plot_solution_comparison: shape '[100, 100]' is invalid for input of size 110000


ERROR:root:Error saving plots: shape '[100, 100]' is invalid for input of size 110000


Training completed successfully.


## 4. Plot prediction vs. reference

After training completes, the latest experiment directory under `results_dir` contains `final_model.pt` and a `live_snapshot.npz`. Reload them and compare a single field channel.

In [5]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

from pinnrl.config import Config, ModelConfig
from pinnrl.neural_networks import PINNModel
from pinnrl.training.train import _build_training_config

results_dir = Path(yaml_cfg["paths"]["results_dir"])
experiment = sorted(results_dir.glob("*active_matter*"))[-1]
print(f"Loading model from {experiment.name}")

# Rebuild ModelConfig to match the trained model and load weights.
arch_type = config_dict["model"]["architecture"]
arch_config = config_dict["architectures"].get(arch_type, {})
config_obj = Config()
config_obj.device = torch.device("cpu")
config_obj.model = ModelConfig(
    input_dim=config_dict["model"]["input_dim"],
    hidden_dim=config_dict["model"].get("hidden_dim", 128),
    output_dim=config_dict["model"]["output_dim"],
    num_layers=config_dict["model"].get("num_layers", 4),
    activation=arch_config.get("activation", "tanh"),
    fourier_features=arch_type == "fourier",
    fourier_scale=arch_config.get("scale", 1.0) if arch_type == "fourier" else None,
    dropout=arch_config.get("dropout", 0.0),
    layer_norm=arch_config.get("layer_norm", False),
    architecture=arch_type,
)
for key in ("mapping_size", "scale", "omega_0", "num_heads",
            "hidden_dims", "latent_dim", "modes", "periodic"):
    if key in arch_config:
        setattr(config_obj.model, key, arch_config[key])
config_obj.training = _build_training_config(config_dict["training"])

model = PINNModel(config=config_obj, device=torch.device("cpu")).to(torch.device("cpu"))
ckpt = torch.load(experiment / "final_model.pt", map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state)
model.eval()

# Predict on the loaded slice and compare per-channel.
x = slice_["x"]; t = slice_["t"]; u_ref = slice_["u"]
with torch.no_grad():
    inp = torch.cat([x, t], dim=1)
    u_pred_np = model(inp).cpu().numpy()
u_ref_np = u_ref.cpu().numpy()
x_np = x.cpu().numpy()

channel_names = ["concentration", "velocity_x", "velocity_y"]
channel_idx = [0, 1, 2]

fig, axes = plt.subplots(2, len(channel_idx), figsize=(4.0 * len(channel_idx), 7.5))
for col, (ch, name) in enumerate(zip(channel_idx, channel_names)):
    ref = u_ref_np[:, ch]
    prd = u_pred_np[:, ch]
    vmin = float(min(ref.min(), prd.min()))
    vmax = float(max(ref.max(), prd.max()))
    sc0 = axes[0, col].scatter(x_np[:, 0], x_np[:, 1], c=ref, cmap="viridis",
                               vmin=vmin, vmax=vmax, s=5)
    axes[0, col].set_title(f"Reference — {name}", fontsize=11, fontweight="bold")
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])
    plt.colorbar(sc0, ax=axes[0, col], shrink=0.8)
    sc1 = axes[1, col].scatter(x_np[:, 0], x_np[:, 1], c=prd, cmap="viridis",
                               vmin=vmin, vmax=vmax, s=5)
    axes[1, col].set_title(f"FNO prediction — {name}", fontsize=11, fontweight="bold")
    axes[1, col].set_xticks([]); axes[1, col].set_yticks([])
    plt.colorbar(sc1, ax=axes[1, col], shrink=0.8)

fig.suptitle("active_matter — Well dataset reference vs FNO prediction (200 epochs, CPU)",
             fontsize=13, fontweight="bold")
fig.tight_layout()

images_dir = Path("images"); images_dir.mkdir(exist_ok=True)
out_path = images_dir / "04_well_active_matter.png"
fig.savefig(out_path, dpi=180, bbox_inches="tight")
print(f"Saved {out_path}")
plt.show()

print("\nChannel-wise error on the loaded slice:")
for ch in range(u_ref_np.shape[1]):
    diff = u_pred_np[:, ch] - u_ref_np[:, ch]
    print(f"  channel {ch:>2}  L2={np.sqrt(np.mean(diff**2)):.3e}  L1={np.mean(np.abs(diff)):.3e}")

Loading model from 20260503_233514_active_matter_fno_no_rl


Saved images/04_well_active_matter.png

Channel-wise error on the loaded slice:
  channel  0  L2=9.190e-03  L1=7.270e-03
  channel  1  L2=5.153e-01  L1=4.228e-01
  channel  2  L2=7.454e-01  L1=6.032e-01
  channel  3  L2=2.661e-01  L1=2.251e-01
  channel  4  L2=2.762e-01  L1=2.351e-01
  channel  5  L2=2.767e-01  L1=2.356e-01
  channel  6  L2=2.662e-01  L1=2.257e-01
  channel  7  L2=4.378e-01  L1=3.610e-01
  channel  8  L2=4.906e-01  L1=4.127e-01
  channel  9  L2=4.913e-01  L1=4.142e-01
  channel 10  L2=4.379e-01  L1=3.614e-01


/var/folders/yx/b_pyg98x1dj97pbflpck32d80000gn/T/ipykernel_75750/1140934795.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Try a matched dataset (`acoustic_scattering_maze`)

For datasets whose default PDE matches an existing pinnrl solver, switch to `data_augmented` to combine the residual loss with the dataset's reference field. This is the regime where pinnrl's RL adaptive sampler can be benchmarked against the ground truth from The Well.

In [ ]:
# Reload the yaml so the previous override (mode=data_only) doesn't leak in;
# build_config_dict only fills mode from the registry when the caller hasn't set one.
matched_yaml = yaml.safe_load(Path("../pinnrl/config/config.yaml").read_text())
matched_yaml.setdefault("training", {}).pop("mode", None)

matched = build_config_dict(
    matched_yaml,
    pde_name="Wave Equation",
    arch_type="fno",
    use_rl=True,
    epochs=300,
    dataset={
        "name": "acoustic_scattering_maze",
        "split": "train",
        "n_traj": 1,
        "n_points": 4_000,
        "seed": 0,
        "base": None,
        "use_defaults": True,
    },
)
print("mode:", matched["training"]["mode"])  # data_augmented (the registry default)
print("observation_data source:", matched["pde"]["observation_data"]["source"])
print("rl_enabled:", matched["rl"]["enabled"])
print("\nThis configuration combines the Well dataset's reference field with pinnrl's PDE\n"
      "residual loss and switches on the DQN sampling agent — the regime where adaptive\n"
      "sampling is benchmarked against ground truth. Run `run_training(matched, device)` to\n"
      "launch it (allow ~10 minutes on CPU).")